# Sigma-term literature comparison

Primary-source values and versions are recorded in `sigma_literature_codex.json`. Errors are combined in quadrature only within a quoted result; no average across publications is performed. Several rows share data. The light comparison labels the explicitly different pion-mass conventions rather than silently converting all results. Our strange result is at one lattice spacing and does not include continuum or finite-volume systematics.

The strange comparison is a manuscript figure. The light and charm comparisons are supplementary previews. This notebook performs no new fits.

In [1]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import util_codex as yc

ROOT = Path.cwd().resolve()
DATA = ROOT / 'sigma_literature_codex.json'
OUTPUT = ROOT / '__codex_ignore/fig/analysis_sigma_literature_codex/internal_ignore'
OUTPUT.mkdir(parents=True, exist_ok=True)
literature = json.loads(DATA.read_text(encoding='utf-8'))
studies = literature['studies']
assert len({row['id'] for row in studies}) == len(studies)
print('Sources checked:', literature['checked_on'])

Sources checked: 2026-09-17


## Comparisons

Results are grouped by method: direct matrix elements (circles), Feynman-Hellmann mass derivatives (squares), chiral reanalyses of sigma-term data (diamonds), and phenomenology (triangles). Open and filled lattice symbols distinguish results without and with a continuum extrapolation. Inner bars show source-quoted statistical errors where separately available; outer bars include all quoted uncertainties. The strange comparison has no legend or grid. Its colors run from red (this work and its band), through green (other direct results) and blue (BMW), to orange (Copeland's chiral fit under Feynman-Hellmann).

The explicit selection in the JSON omits superseded results and the very noisy RQCD strange determination. Only ETMC's continuum result is plotted; its B64 value is retained in the source records. The two Kou-Chen photoproduction records are retained but not plotted: their quoted energy-dependent spreads exclude model uncertainties, which are not fully quantified. Historical source records remain available. Copeland takes mass derivatives of chiral fits to published baryon masses without a continuum extrapolation; its total error includes the third, theoretical component from Table I. The two PNDME light rows are alternative analyses of the same data. Liang's chiral reanalysis instead fits Mainz sigma-term data. The continuum marker does not imply a uniform systematic error budget.

In [2]:
for channel in ['piN', 's', 'c']:
    figure, axis = yc.plot_sigma_literature(studies, channel, literature['plot_groups'][channel])
    figure.savefig(OUTPUT / f'sigma_{channel}_literature.pdf')
    figure.savefig(OUTPUT / f'sigma_{channel}_literature.png', dpi=200)
    plt.show()
    plt.close(figure)

C:\Users\yan14\AppData\Local\Temp\ipykernel_101360\2333568117.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\yan14\AppData\Local\Temp\ipykernel_101360\2333568117.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\yan14\AppData\Local\Temp\ipykernel_101360\2333568117.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Source table and validation

The table preserves component errors in the JSON and reports the combined interval in MeV here. Source-quoted totals take precedence over recombining rounded components (Mainz). BMW values are converted from scalar fractions with the source's nucleon-mass normalization, 938.919 MeV. RQCD's strange errors are asymmetric total intervals, not two separate uncertainty components.

In [3]:
import gvar as gv
from IPython.display import Markdown, display

lines = ['| Source | Channel | Result [MeV] | Location |',
         '|---|---|---|---|']
for row in studies:
    for channel in row['values']:
        mean, minus, plus = yc.sigma_literature_values(row, channel)
        result = str(gv.gvar(mean, minus)) if minus == plus else f'{mean:g} (+{plus:g}/-{minus:g})'
        source = f"[{row['label']}]({row['source']})" if row['source'].startswith('https:') else row['label']
        lines.append(f"| {source} | {channel} | {result} | {row['locator']} |")
lines.extend(['', '## Collected separately'])
for row in literature.get('collected_not_plotted', []):
    lines.append(f"\n[{row['label']}]({row['source']}): {row['reason']}")
    for method in ['exponential', 'dipole']:
        mean, error = row[method]
        lines.append(f"- {method}: {gv.gvar(mean, error)} MeV")
report = '\n'.join(lines)
(OUTPUT / 'source_table.md').write_text(report + '\n', encoding='utf-8')
display(Markdown(report))

rqcd = next(row for row in studies if row['id'] == 'rqcd23')
assert yc.sigma_literature_values(rqcd, 's') == (16, 68, 58)
bmw = next(row for row in studies if row['id'] == 'bmw20')
assert abs(yc.sigma_literature_values(bmw, 'c')[0] - .0734 * 938.919) < 1e-12
assert not any(channel in studies[0]['values'] for channel in ['piN', 'c'])
print('Validated source IDs, asymmetric errors, unit conversion, and our quoted channel.')

| Source | Channel | Result [MeV] | Location |
|---|---|---|---|
| [Kou-Chen (2024), exp.](https://doi.org/10.1103/PhysRevD.109.036034) | s | 456(65) | Abstract and Sec. III.A, paragraph following Fig. 2 |
| [Kou-Chen (2024), dipole](https://doi.org/10.1103/PhysRevD.109.036034) | s | 3.4(1.1)e+02 | Abstract and Sec. III.A, paragraph following Fig. 2 |
| [ETMC (2025), B64](https://arxiv.org/abs/2412.01535v1) | s | 56(12) | Table VIII, B64 row |
| This work | s | 43.9(2.6) | AIC constant-fit average |
| [PNDME (2025)](https://arxiv.org/abs/2503.07100v3) | piN | 61.0(6.0) | Eq. (10), final published version 19 December 2025 |
| [PNDME (2025)](https://arxiv.org/abs/2503.07100v3) | s | 35(13) | Eq. (10), final published version 19 December 2025 |
| [PNDME (2025), standard](https://arxiv.org/abs/2503.07100v3) | piN | 42.0(6.0) | Eq. (10) |
| [ETMC (2025)](https://arxiv.org/abs/2412.01535v1) | piN | 41.9(8.1) | Table VIII, continuum row; comparison Fig. 17 |
| [ETMC (2025)](https://arxiv.org/abs/2412.01535v1) | s | 30(17) | Table VIII, continuum row; comparison Fig. 17 |
| [ETMC (2025)](https://arxiv.org/abs/2412.01535v1) | c | 82(29) | Table VIII, continuum row; comparison Fig. 17 |
| [Mainz (2023)](https://arxiv.org/abs/2303.08741v2) | piN | 43.7(3.6) | Eq. (30) |
| [Mainz (2023)](https://arxiv.org/abs/2303.08741v2) | s | 28.6(9.3) | Eq. (30) |
| [RQCD (2023)](https://arxiv.org/abs/2211.03744v2) | piN | 43.9(4.7) | Table 10, nucleon row, sigma (not tilde sigma) columns |
| [RQCD (2023)](https://arxiv.org/abs/2211.03744v2) | s | 16 (+58/-68) | Table 10, nucleon row, sigma (not tilde sigma) columns |
| [PNDME (2021)](https://arxiv.org/abs/2105.12095v2) | piN | 59.6(7.4) | Abstract, preferred N pi analysis |
| [BMW (2020)](https://arxiv.org/abs/2007.03319v1) | piN | 37.4(5.1) | Table 1, f_ud^N, f_s^N, f_c^N; nucleon mass SM Sec.3 |
| [BMW (2020)](https://arxiv.org/abs/2007.03319v1) | s | 54.2(5.3) | Table 1, f_ud^N, f_s^N, f_c^N; nucleon mass SM Sec.3 |
| [BMW (2020)](https://arxiv.org/abs/2007.03319v1) | c | 68.9(6.7) | Table 1, f_ud^N, f_s^N, f_c^N; nucleon mass SM Sec.3 |
| [ETMC (2020)](https://arxiv.org/abs/1909.00485v2) | piN | 41.6(3.8) | Table XII |
| [ETMC (2020)](https://arxiv.org/abs/1909.00485v2) | s | 45.6(6.2) | Table XII |
| [ETMC (2020)](https://arxiv.org/abs/1909.00485v2) | c | 107(22) | Table XII |
| [chiQCD (2016)](https://arxiv.org/abs/1511.09089v3) | piN | 45.9(7.9) | Abstract and final results |
| [chiQCD (2016)](https://arxiv.org/abs/1511.09089v3) | s | 40(12) | Abstract and final results |
| [chiQCD (2013), heavy sea](https://arxiv.org/abs/1304.1194v4) | c | 94(31) | Abstract and charm conclusion |
| [Copeland et al. (2023)](https://arxiv.org/abs/2112.03198v2) | piN | 44.0(5.8) | Table I, nucleon row and caption; Sec. III on absence of a continuum limit |
| [Copeland et al. (2023)](https://arxiv.org/abs/2112.03198v2) | s | 50(12) | Table I, nucleon row and caption; Sec. III on absence of a continuum limit |
| [Liang et al. (2025), neutral](https://arxiv.org/abs/2508.11435v1) | piN | 53.1(2.4) | After Fig.4, neutral-pion total error; Table S2, neutral-pion statistical error |
| [Roy-Steiner (2023), isoQCD](https://arxiv.org/abs/2305.07045v2) | piN | 55.9(3.5) | Updated sigma term and isospin correction |
| [Scattering (2018), charged](https://arxiv.org/abs/1706.01465v2) | piN | 58.0(5.0) | Abstract |
| [Pionic atoms (2019), quoted](https://arxiv.org/abs/1901.03130v2) | piN | 57.0(7.0) | Abstract |

## Collected separately

Validated source IDs, asymmetric errors, unit conversion, and our quoted channel.
